# Sydney Rain Forecasting Experiment

This notebook is the professional experiment record for the weather forecasting project. It keeps narrative, diagnostics, and model refresh steps in one place while delegating reusable logic to `src/weather_forecasting`.

## 1. Problem Framing

The product goal is to support operational planning for weather-sensitive decisions in Sydney. The model suite answers two questions from the latest complete daily observation:

- Will measurable rain occur 7 days from now?
- How much precipitation is expected across the next 3 days?

The classification model is evaluated with threshold-aware precision, recall, F1, ROC-AUC, and average precision. The regression model is evaluated with MAE, RMSE, and R2.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd

from weather_forecasting.config import PROCESSED_DATA_DIR
from weather_forecasting.inference import predict_latest
from weather_forecasting.models import load_design_matrices, train_and_save_models

PROJECT_ROOT

## 2. Data Inventory

The project uses Open-Meteo daily weather data refreshed through yesterday in Sydney time. The training function rebuilds the processed dataset, design matrices, model bundles, and metadata from the same reusable project code used by the dashboard.

In [ ]:
processed_files = sorted(PROCESSED_DATA_DIR.glob('*'))
pd.DataFrame({
    'file': [p.name for p in processed_files],
    'size_mb': [round(p.stat().st_size / 1_000_000, 3) for p in processed_files],
})

In [ ]:
daily = pd.read_csv(PROCESSED_DATA_DIR / 'sydney_daily_current.csv', parse_dates=['date'])
daily[['date', 'precipitation_sum', 'rain_sum', 'temperature_2m_mean', 'pressure_msl_mean']].describe(include='all')

## 3. Feature Schema

The live app depends on the saved feature order. This check confirms that classification and regression use the same 47-feature contract.

In [ ]:
matrices = load_design_matrices()
schema = pd.DataFrame({
    'classification': matrices['X_train_cls'].columns,
    'regression': matrices['X_train_reg'].columns,
})
schema

## 4. Regenerate Model Bundles

Rerun this cell whenever the design matrices or modeling approach changes. It refreshes the local `joblib` files, `features.txt`, and deployment metadata used by Streamlit.

In [ ]:
results = train_and_save_models(refresh_data=True)
results

## 5. Live Prediction Smoke Test

This verifies that the freshly saved bundles can be loaded and used by the same inference function as the Streamlit app.

In [ ]:
predict_latest()

## 6. Deployment Notes

Run the dashboard locally with:

```powershell
streamlit run streamlit_app.py
```

Next model improvements should focus on stronger temporal validation, recalibrating the rain threshold around business costs, and replacing current-day observed weather features with true forecast inputs for a stricter production forecast.